In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

### Customers Data

**Read Bronze**

In [0]:
customers_bronze_df = spark.table(
    "workspace.bronze.company_a_customers"
)

**Type casting/Creating Silver Version**

In [0]:
customers_silver_df = (
    customers_bronze_df
    .select(
        col("customer_id"),
        col("customer_name"),
        col("email"),
        col("city"),
        col("state"),
        to_date(
            col("signup_date"),
            "yyyy-MM-dd"
        ).alias("signup_date"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Validate**

In [0]:
customers_silver_df.count()

In [0]:
customers_silver_df \
    .groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
customers_silver_df.filter(
    col("customer_id").isNull()
    | col("customer_name").isNull()
    | col("signup_date").isNull()
).count()

**Write to silver**

In [0]:
(
    customers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_a_customers"
    )
)

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM workspace.silver.company_a_customers
""").show()

### Products Data

**Read Bronze**

In [0]:
products_bronze_df = spark.table(
    "workspace.bronze.company_a_products"
)

**Type casting/Creating Silver Version**

In [0]:
products_silver_df = (
    products_bronze_df
    .select(
        col("product_id"),
        col("product_name"),
        col("category"),
        col("unit_price")
            .cast(DecimalType(10, 2))
            .alias("unit_price"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Validate**

In [0]:
products_silver_df.count()

In [0]:
(
    products_silver_df
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
products_silver_df.filter(
    col("product_id").isNull()
    | col("product_name").isNull()
    | col("category").isNull()
    | col("unit_price").isNull()
).count()

In [0]:
products_silver_df.filter(
    col("unit_price") <= 0
).count()

**Write to Silver**

In [0]:
(
    products_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_a_products"
    )
)

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS product_count
FROM workspace.silver.company_a_products
""").show()

### Orders Data

**Read Bronze**

In [0]:
orders_bronze_df = spark.table(
    "workspace.bronze.company_a_orders"
)

**Type Casting/Silver Version**

In [0]:
from pyspark.sql.functions import col, to_date

orders_silver_df = (
    orders_bronze_df
    .select(
        col("order_id"),
        col("customer_id"),
        to_date(
            col("order_date"),
            "yyyy-MM-dd"
        ).alias("order_date"),
        col("order_status"),
        col("payment_method"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Validate**

In [0]:
orders_silver_df.count()

In [0]:
(
    orders_silver_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
orders_silver_df.filter(
    col("order_id").isNull()
    | col("customer_id").isNull()
    | col("order_date").isNull()
    | col("order_status").isNull()
).count()

In [0]:
customers_silver_df = spark.table(
    "workspace.silver.company_a_customers"
)

In [0]:
invalid_customer_orders_df = (
    orders_silver_df.alias("o")
    .join(
        customers_silver_df
        .select("customer_id")
        .alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left_anti",
    )
)

In [0]:
invalid_customer_orders_df.count()

In [0]:
orders_silver_df \
    .groupBy("order_status") \
    .count() \
    .show()

In [0]:
orders_silver_df \
    .groupBy("payment_method") \
    .count() \
    .show()

**Write to Silver**

In [0]:
(
    orders_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_a_orders"
    )
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS order_count
FROM workspace.silver.company_a_orders
""").show()

### Orders Items

**Read Bronze**

In [0]:
order_items_bronze_df = spark.table(
    "workspace.bronze.company_a_order_items"
)

**Type Casting/Silver Version**

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType, IntegerType

order_items_silver_df = (
    order_items_bronze_df
    .select(
        col("order_item_id"),
        col("order_id"),
        col("product_id"),
        col("quantity")
            .cast(IntegerType())
            .alias("quantity"),
        col("unit_price")
            .cast(DecimalType(10, 2))
            .alias("unit_price"),
        col("discount_pct")
            .cast(DecimalType(5, 2))
            .alias("discount_pct"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Validate**

In [0]:
order_items_silver_df.count()

In [0]:
(
    order_items_silver_df
    .groupBy("order_item_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

In [0]:
order_items_silver_df.filter(
    col("order_item_id").isNull()
    | col("order_id").isNull()
    | col("product_id").isNull()
    | col("quantity").isNull()
    | col("unit_price").isNull()
    | col("discount_pct").isNull()
).count()

In [0]:
order_items_silver_df.filter(
    (col("quantity") <= 0)
    | (col("unit_price") <= 0)
    | (col("discount_pct") < 0)
    | (col("discount_pct") > 100)
).count()

In [0]:
orders_silver_df = spark.table(
    "workspace.silver.company_a_orders"
)

invalid_order_refs_df = (
    order_items_silver_df.alias("oi")
    .join(
        orders_silver_df
        .select("order_id")
        .alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "left_anti",
    )
)

invalid_order_refs_df.count()

In [0]:
products_silver_df = spark.table(
    "workspace.silver.company_a_products"
)

invalid_product_refs_df = (
    order_items_silver_df.alias("oi")
    .join(
        products_silver_df
        .select("product_id")
        .alias("p"),
        col("oi.product_id") == col("p.product_id"),
        "left_anti",
    )
)

invalid_product_refs_df.count()

**Write to Silver**

In [0]:
(
    order_items_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_a_order_items"
    )
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS order_item_count
FROM workspace.silver.company_a_order_items
""").show()